# Notebook 01 — Exploratory Data Analysis: Open-Meteo Historical Weather

This notebook walks through a complete EDA of the Open-Meteo daily weather
data that was downloaded into MinIO by the `download_weather_datasets` Airflow DAG.

**What you will learn**
- How to read a CSV from MinIO into a pandas DataFrame
- Basic data quality checks: null counts, dtypes, value ranges
- Descriptive statistics with `describe()` and `groupby()`
- Time-series resampling to monthly and annual aggregates
- Visualising distributions (histogram), trends (line chart), and
  comparisons (box plot, heatmap) with matplotlib and seaborn
- Running SQL directly on a DataFrame using DuckDB

**Pre-requisites**
- The `download_weather_datasets` DAG must have run at least once so that
  `weather-raw/open-meteo/new_york.csv` exists in MinIO.
- The Jupyter image must have `matplotlib` and `seaborn` installed
  (see `apps/datascience/jupyter/Containerfile` — add them if not present).

---

In [ ]:
# Cell 1 — Install any missing packages
# Run this cell once if matplotlib or seaborn are not yet in the image.
# After adding them to the Containerfile, rebuild the image instead.
import importlib

for pkg in ['matplotlib', 'seaborn']:
    if importlib.util.find_spec(pkg) is None:
        print(f'Installing {pkg}...')
        import subprocess, sys
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '--quiet'])
    else:
        print(f'{pkg} already installed')

In [ ]:
# Cell 2 — Imports
import sys
import os

# The shared helpers are mounted into /home/jovyan/work/shared
# (copy the files there or adjust the path to wherever you placed them)
SHARED_PATH = '/home/jovyan/work/shared'
if SHARED_PATH not in sys.path:
    sys.path.insert(0, SHARED_PATH)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import duckdb

from minio_helper import get_client, read_csv, object_exists

# Make plots render inline in JupyterLab
%matplotlib inline

# Consistent figure style
sns.set_theme(style='whitegrid', palette='tab10')
plt.rcParams['figure.dpi'] = 120

print('Imports OK')

In [ ]:
# Cell 3 — Load data from MinIO
# DATA: We read a single-location CSV to keep the initial exploration simple.
#       Later cells stack all five locations for comparison.

BUCKET = 'weather-raw'
OBJECT = 'open-meteo/new_york.csv'

client = get_client()

if not object_exists(client, BUCKET, OBJECT):
    raise FileNotFoundError(
        f'{BUCKET}/{OBJECT} not found in MinIO. '
        'Run the download_weather_datasets Airflow DAG first.'
    )

df = read_csv(
    client,
    BUCKET,
    OBJECT,
    # Parse the 'date' column as a datetime so time-series operations work
    parse_dates=['date'],
)

print(f'Shape: {df.shape}')
df.head()

In [ ]:
# Cell 4 — Data quality checks
# Always do this before any analysis. It tells you:
#   - Which columns have missing values (and how many)
#   - Whether dtypes are as expected
#   - The date range covered

print('=== dtypes ===')
print(df.dtypes)
print()

print('=== null counts ===')
null_counts = df.isnull().sum()
print(null_counts[null_counts > 0] if null_counts.any() else 'No nulls found')
print()

print('=== date range ===')
print(f'From: {df["date"].min().date()}')
print(f'To:   {df["date"].max().date()}')
print(f'Days: {(df["date"].max() - df["date"].min()).days + 1}')
print(f'Rows: {len(df)}')

# DATA assumption: the number of rows should equal the number of calendar days.
# If it is less, there are gaps in the data.
expected_rows = (df['date'].max() - df['date'].min()).days + 1
if len(df) < expected_rows:
    print(f'WARNING: {expected_rows - len(df)} days missing from the series')

In [ ]:
# Cell 5 — Descriptive statistics
# describe() gives count, mean, std, min, quartiles, max for every
# numeric column. This is the fastest way to spot unreasonable values.

df.describe().round(2)

In [ ]:
# Cell 6 — Temperature distribution histogram
# A histogram shows the frequency of each temperature range.
# For New York we expect a bimodal-ish distribution because summers
# and winters are very different.

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, col, label in [
    (axes[0], 'temperature_2m_max', 'Daily Max Temp (°C)'),
    (axes[1], 'temperature_2m_min', 'Daily Min Temp (°C)'),
    (axes[2], 'temperature_2m_mean', 'Daily Mean Temp (°C)'),
]:
    ax.hist(df[col].dropna(), bins=40, edgecolor='white', linewidth=0.5)
    ax.set_title(f'Distribution of {label}', fontsize=11)
    ax.set_xlabel(label)
    ax.set_ylabel('Number of days')

fig.suptitle('New York — Temperature Distributions (2020–2024)', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Cell 7 — Time-series line chart: monthly mean temperature
# Resampling collapses daily rows into monthly summaries.
# resample('ME') uses month-end frequency (pandas >= 2.2 alias).
# PERF: resample is fast for this dataset size (~1800 rows).

# Set the date column as the index so resample() can use it
df_indexed = df.set_index('date')

monthly = df_indexed[['temperature_2m_max', 'temperature_2m_min', 'temperature_2m_mean']].resample('ME').mean()

fig, ax = plt.subplots(figsize=(12, 4))

ax.plot(monthly.index, monthly['temperature_2m_max'],  label='Monthly avg of daily max', linewidth=1.5)
ax.plot(monthly.index, monthly['temperature_2m_mean'], label='Monthly avg of daily mean', linewidth=1.5)
ax.plot(monthly.index, monthly['temperature_2m_min'],  label='Monthly avg of daily min', linewidth=1.5)

# Shade the range between min and max to show temperature spread
ax.fill_between(
    monthly.index,
    monthly['temperature_2m_min'],
    monthly['temperature_2m_max'],
    alpha=0.15,
    label='Min–Max range',
)

ax.set_title('New York — Monthly Temperature (2020–2024)', fontsize=13)
ax.set_xlabel('Month')
ax.set_ylabel('Temperature (°C)')
ax.legend()
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Cell 8 — Monthly precipitation bar chart
# Sum (not mean) is the right aggregation for precipitation.

monthly_precip = df_indexed[['precipitation_sum']].resample('ME').sum()

fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(monthly_precip.index, monthly_precip['precipitation_sum'], width=20, align='center')
ax.set_title('New York — Monthly Total Precipitation (2020–2024)', fontsize=13)
ax.set_xlabel('Month')
ax.set_ylabel('Precipitation (mm)')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Cell 9 — Box plot: temperature by calendar month
# Box plots show median, IQR, and outliers. This tells us which months
# are most variable, not just their average.

# Add a 'month' column (integer 1–12) for grouping
df['month'] = df['date'].dt.month
month_names = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

fig, ax = plt.subplots(figsize=(12, 5))

# seaborn boxplot: x=month, y=temperature, one box per month
sns.boxplot(
    data=df,
    x='month',
    y='temperature_2m_mean',
    ax=ax,
    # Each box is coloured by month — purely cosmetic
    hue='month',
    palette='coolwarm',
    legend=False,
)

ax.set_xticklabels(month_names)
ax.set_title('New York — Daily Mean Temperature Distribution by Month (2020–2024)', fontsize=13)
ax.set_xlabel('Month')
ax.set_ylabel('Daily Mean Temperature (°C)')
plt.tight_layout()
plt.show()

In [ ]:
# Cell 10 — Correlation heatmap
# A heatmap of the correlation matrix shows which variables move together.
# Strong positive correlations appear dark red; strong negative appear dark blue.

# Select numeric columns only
numeric_cols = [
    'temperature_2m_max', 'temperature_2m_min', 'temperature_2m_mean',
    'precipitation_sum', 'wind_speed_10m_max', 'shortwave_radiation_sum',
]

# Only use columns that exist in the DataFrame
available = [c for c in numeric_cols if c in df.columns]
corr = df[available].corr()

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    corr,
    annot=True,       # show the correlation coefficient in each cell
    fmt='.2f',
    cmap='RdBu_r',    # red=positive, blue=negative (colorblind-accessible)
    vmin=-1, vmax=1,
    ax=ax,
)
ax.set_title('New York — Weather Variable Correlation Matrix', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Cell 11 — Multi-location comparison
# Load all five locations and stack them into one DataFrame for comparison.
# This demonstrates reading multiple objects from MinIO in a loop.

LOCATIONS = ['new_york', 'london', 'tokyo', 'melbourne', 'singapore']

frames = []
for loc in LOCATIONS:
    obj = f'open-meteo/{loc}.csv'
    if object_exists(client, BUCKET, obj):
        loc_df = read_csv(client, BUCKET, obj, parse_dates=['date'])
        frames.append(loc_df)
    else:
        print(f'WARNING: {obj} not found in MinIO — skipping {loc}')

if frames:
    all_df = pd.concat(frames, ignore_index=True)
    print(f'Combined dataset: {all_df.shape}')
    all_df.groupby('location')[['temperature_2m_mean', 'precipitation_sum']].describe().round(1)
else:
    print('No data loaded. Run the Airflow DAG first.')

In [ ]:
# Cell 12 — Annual mean temperature by location (pivot_table)
# pivot_table is like a spreadsheet pivot: rows=year, columns=location,
# values=mean temperature. Great for spotting which city is warmest/coolest
# year-over-year.

if 'all_df' in dir():
    all_df['year'] = all_df['date'].dt.year

    pivot = all_df.pivot_table(
        index='year',
        columns='location',
        values='temperature_2m_mean',
        aggfunc='mean',
    ).round(1)

    display(pivot)

    # Heatmap of the pivot table
    fig, ax = plt.subplots(figsize=(10, 4))
    sns.heatmap(
        pivot.T,   # transpose so locations are rows, years are columns
        annot=True,
        fmt='.1f',
        cmap='RdYlBu_r',
        ax=ax,
    )
    ax.set_title('Annual Mean Temperature by Location (°C)', fontsize=13)
    ax.set_xlabel('Year')
    ax.set_ylabel('Location')
    plt.tight_layout()
    plt.show()

In [ ]:
# Cell 13 — DuckDB SQL analysis
# DuckDB can query a pandas DataFrame directly using SQL.
# This is useful when the query is more natural to express in SQL than
# in pandas (e.g., window functions, CTEs).

# Create an in-memory DuckDB connection
con = duckdb.connect(':memory:')

# Register the DataFrame as a virtual table named 'weather'
# DATA: DuckDB's register() takes a name and a pandas DataFrame.
# The docs confirm this API: https://duckdb.org/docs/api/python/overview.html
con.register('weather', all_df if 'all_df' in dir() else df)

# Query: top 3 hottest days per location
result = con.execute("""
    SELECT
        location,
        date,
        temperature_2m_max,
        RANK() OVER (PARTITION BY location ORDER BY temperature_2m_max DESC) AS rank
    FROM weather
    WHERE temperature_2m_max IS NOT NULL
    QUALIFY rank <= 3
    ORDER BY location, rank
""").df()  # .df() returns a pandas DataFrame

result

In [ ]:
# Cell 14 — Monthly precipitation comparison: grouped bar chart
# groupby + unstack reshapes the data from long to wide format.

if 'all_df' in dir():
    all_df['month'] = all_df['date'].dt.month

    monthly_precip_loc = (
        all_df.groupby(['location', 'month'])['precipitation_sum']
        .mean()           # average daily precipitation for that month
        .unstack(level=0) # pivot locations into columns
    )

    fig, ax = plt.subplots(figsize=(13, 5))
    monthly_precip_loc.plot(kind='bar', ax=ax, width=0.8)

    ax.set_title('Average Daily Precipitation by Month and Location (2020–2024)', fontsize=13)
    ax.set_xlabel('Month')
    ax.set_ylabel('Avg Daily Precipitation (mm)')
    ax.set_xticklabels(month_names, rotation=0)
    ax.legend(title='Location', bbox_to_anchor=(1.01, 1), loc='upper left')
    plt.tight_layout()
    plt.show()